# Candidate: Cython-accelerated Betweenness Centrality

**Repo:** `networkx/networkx`  
**Baseline tag:** `networkx-2.8`  
**Optimization:** CSR adjacency + typed C arrays in Cython (see `betweenness_core.pyx`)

In [ ]:
!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h
!python --version

In [ ]:
# ── Install NetworkX 2.8 and build deps ──────────────────────────────────────
!rm -rf nx_baseline
!git clone --depth 1 --branch networkx-2.8 https://github.com/networkx/networkx.git nx_baseline
!pip install nx_baseline/ --quiet 2>&1 | grep -E 'Successfully|ERROR|error' || true
!pip install cython numpy --quiet 2>&1 | grep -E 'Successfully|already|ERROR' || true

# Flush any cached networkx so Python picks up the 2.8 install
import sys
for k in list(sys.modules):
    if k.startswith('networkx'):
        del sys.modules[k]

import networkx as nx
print('NetworkX version:', nx.__version__)   # must be 2.8
assert nx.__version__ == '2.8', f'Wrong version: {nx.__version__}'

In [ ]:
# ── Clone submission repo and build Cython extension ─────────────────────────
# NOTE: we do NOT os.chdir:we stay in /content the whole notebook
# so that the networkx install on sys.path stays visible.
import subprocess, sys, os

REPO_URL = 'https://github.com/deepsodha/networkx-betweenness-opt.git'
!rm -rf submission_repo
!git clone {REPO_URL} submission_repo

# Build the Cython extension inside the repo folder
result = subprocess.run(
    [sys.executable, 'setup.py', 'build_ext', '--inplace'],
    cwd='submission_repo',
    capture_output=True, text=True
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('Build failed!')
print('Build complete.')

# Add the repo folder to sys.path so we can import betweenness_core
REPO_PATH = os.path.abspath('submission_repo')
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from betweenness_core import betweenness_centrality_cython
print('betweenness_core imported successfully.')

## What was changed and why is it faster?

### The slow path

NetworkX 2.8's `betweenness_centrality` calls two helper functions per source node:

- **`_single_source_shortest_path_basic(G, s)`**:BFS using a Python `deque`,
  with `sigma`, `D` (distance), and `P` (predecessors) stored as Python `dict`s.
  Every `dict.__getitem__`, `dict.__setitem__`, and `deque.append` is a Python
  bytecode dispatch plus reference-count update.

- **`_accumulate_basic(betweenness, S, P, sigma, s)`**:back-propagation loop
  (Brandes dependency accumulation) also operating on Python dicts.

With n=2000 nodes and ~12 000 edges, these run **2000 times each**:roughly
28 million Python-level operations total.

### What we changed

We wrote `betweenness_core.pyx`, a Cython module with `betweenness_centrality_cython(G)`:

1. **CSR representation** : graph converted once to integer-indexed `int32` arrays;
   neighbour lookups become C array reads instead of Python dict accesses.
2. **Typed C arrays** : `sigma`, `dist`, `delta`, `queue`, `stack` declared as
   `cnp.ndarray[FLOAT64/INT32]` memoryviews; all reads/writes compile to pointer arithmetic.
3. **Compiled BFS inner loop** : the neighbour iteration compiles to a tight C for-loop;
   each edge visit is ~5 C instructions vs ~50+ Python bytecodes.
4. **Compiled back-propagation** : stack-based accumulation also uses typed arrays.
5. **Identical rescaling** : replicates `_rescale` from nx 2.8 exactly.

### Trade-offs

| Gain | Cost |
|---|---|
| ~11x speedup | Requires Cython + GCC at build time (~15s) |
| Zero correctness error (< 1e-18 abs) | Only unweighted, non-sampled, no-endpoints case |
| Same public API | Weighted/approximate variants still use nx 2.8 path |

In [ ]:
import json, time
import networkx as nx
from betweenness_core import betweenness_centrality_cython

print('NetworkX version:', nx.__version__)

SEED = 42
G = nx.barabasi_albert_graph(n=2000, m=6, seed=SEED)
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Run candidate and save output
candidate_bc = betweenness_centrality_cython(G)

with open('candidate_output.json', 'w') as f:
    json.dump({str(k): v for k, v in candidate_bc.items()}, f)

print('Candidate output saved.')
for node, val in sorted(candidate_bc.items(), key=lambda x: -x[1])[:5]:
    print(f'  node {node}: {val:.6f}')

*(Correctness check and speedup calculation are in `tests.ipynb` see there for reward.json.)*